# Notebook 04 — Layer 2B Experiments
Train XGBoost, 1D CNN, and GRU. Compare on macro-F1 and per-class F1. Export winner to ONNX.

> Use GPU runtime in Colab for CNN and GRU training.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import mlflow
from pathlib import Path

from layer2b.candidates.xgboost_model import XGBoostModel
from layer2b.candidates.cnn_1d import CNN1DModel
from layer2b.candidates.gru import GRUModel
from layer2b.evaluate import evaluate_candidate, pick_best
from evaluation.compare_models import compare_l2b, pick_best_l2b
from evaluation.benchmark import benchmark_xgboost, benchmark_cnn_1d, benchmark_gru

SPLITS  = Path("../data/splits")
EXPORTS = Path("../exported_models")
EXPORTS.mkdir(exist_ok=True)

CLASS_NAMES = ["normal", "sqli", "xss", "lfi", "other_attack"]

mlflow.set_experiment("layer2b")

## 1. Load splits

In [ ]:
X_tr_num  = np.load(SPLITS / "l2b_train_X_numeric.npy")
X_v_num   = np.load(SPLITS / "l2b_val_X_numeric.npy")
X_te_num  = np.load(SPLITS / "l2b_test_X_numeric.npy")

X_tr_tok  = np.load(SPLITS / "l2b_train_X_tokens.npy")
X_v_tok   = np.load(SPLITS / "l2b_val_X_tokens.npy")
X_te_tok  = np.load(SPLITS / "l2b_test_X_tokens.npy")

y_tr = np.load(SPLITS / "l2b_train_y.npy")
y_v  = np.load(SPLITS / "l2b_val_y.npy")
y_te = np.load(SPLITS / "l2b_test_y.npy")

import numpy as np
print(f"Train: {X_tr_num.shape}  classes: {dict(zip(*np.unique(y_tr, return_counts=True)))}")
print(f"Val:   {X_v_num.shape}")
print(f"Test:  {X_te_num.shape}")

## 2. Candidate 1 — XGBoost (numeric features, CPU)

In [ ]:
xgb = XGBoostModel()
xgb.train(X_tr_num, y_tr, X_v_num, y_v)
res_xgb = evaluate_candidate(xgb, X_te_num, y_te, name="xgboost")

### XGBoost feature importance

In [ ]:
from feature_engineering.extractor import FEATURE_NAMES
import matplotlib.pyplot as plt

fi = xgb.feature_importance(FEATURE_NAMES)
plt.figure(figsize=(10, 6))
plt.barh(list(fi.keys())[:15], list(fi.values())[:15], color="steelblue")
plt.xlabel("Importance")
plt.title("XGBoost feature importance (top 15)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../data/processed/04_xgb_feature_importance.png", dpi=120)
plt.show()

## 3. Candidate 2 — 1D CNN (character sequences, GPU)

In [ ]:
cnn = CNN1DModel()
cnn.train(X_tr_tok, y_tr, X_v_tok, y_v)
res_cnn = evaluate_candidate(cnn, X_te_tok, y_te, name="cnn_1d")

## 4. Candidate 3 — GRU (character sequences, GPU)

In [ ]:
gru = GRUModel()
gru.train(X_tr_tok, y_tr, X_v_tok, y_v)
res_gru = evaluate_candidate(gru, X_te_tok, y_te, name="gru")

## 5. Compare all candidates

In [ ]:
all_results = [res_xgb, res_cnn, res_gru]
df_compare = compare_l2b(all_results)
df_compare

## 6. Pick winner and export ONNX

In [ ]:
all_models = {"xgboost": xgb, "cnn_1d": cnn, "gru": gru}
winner_name, winner_model = pick_best_l2b(all_results, all_models)
print(f"Winner: {winner_name}")

onnx_path = str(EXPORTS / "layer2b_best.onnx")
winner_model.export_onnx(onnx_path)

## 7. Confusion matrices — all models

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model, X_te_input) in zip(axes, [
    ("XGBoost",  xgb, X_te_num),
    ("CNN 1D",   cnn, X_te_tok),
    ("GRU",      gru, X_te_tok),
]):
    preds = model.predict(X_te_input)
    cm    = confusion_matrix(y_te, preds, labels=list(range(5)))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.savefig("../data/processed/04_confusion_matrices.png", dpi=120)
plt.show()

## 8. GRU attention visualisation (single anomalous request)

In [ ]:
from feature_engineering.tokenizer import CharTokenizer

tok = CharTokenizer()

# pick one attack sample
attack_idx = np.where(y_te > 0)[0][0]
token_ids  = X_te_tok[attack_idx:attack_idx+1]

char_weights = gru.attention_map(token_ids, tok)

# show top 20 highest-attention characters
top20 = sorted(char_weights, key=lambda x: x[1], reverse=True)[:20]
print("Top 20 attention characters:")
for ch, w in top20:
    bar = "#" * int(w * 500)
    print(f"  '{ch}' {bar} ({w:.4f})")